# BrainAI on Colab (GPU)
Runtime > Change runtime type > T4 GPU. Everything runs on the session's local disk; download the checkpoint zip at the end.

In [1]:
# 1) repo + dependencies (no Google login needed)
!apt-get -qq install git-lfs && git lfs install
!git clone https://github.com/ZahidServers/BrainAI.git /content/BrainAI
%cd /content/BrainAI
!pip -q install torch datasets pillow opencv-python-headless transformers accelerate

Selecting previously unselected package git-lfs.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../git-lfs_3.4.1-1ubuntu0.4_amd64.deb ...
Unpacking git-lfs (3.4.1-1ubuntu0.4) ...
Setting up git-lfs (3.4.1-1ubuntu0.4) ...
Processing triggers for man-db (2.12.0-4build2) ...
Git LFS initialized.
Cloning into '/content/BrainAI'...
remote: Enumerating objects: 185, done.
remote: Counting objects: 100% (185/185), done.
remote: Compressing objects: 100% (143/143), done.
remote: Total 185 (delta 38), reused 178 (delta 35), pack-reused 0 (from 0)
Receiving objects: 100% (185/185), 40.17 MiB | 18.46 MiB/s, done.
Resolving deltas: 100% (38/38), done.
Updating files: 100% (158/158), done.
/content/BrainAI


In [2]:
# 2) storage: this session's local disk (no Google Drive needed)
import os
D = '/content/brainai'
os.makedirs(D + '/data', exist_ok=True)
print('storage:', D)

storage: /content/brainai


## 1. Text corpora (reading/writing)

In [3]:
!python tools/fetch_text.py --out $D/data --mb 500
!ln -sf $D/data/*.txt data/   # corpus.py auto-discovers every data/*.txt as a domain

README.md: 100% 26.4k/26.4k [00:00<00:00, 10.6MB/s]
Resolving data files: 100% 2410/2410 [00:00<00:00, 24212.30it/s]
[done] fineweb_edu: 500 MB -> /content/brainai/data/corpus_fineweb_edu.txt
README.md: 100% 131k/131k [00:00<00:00, 106MB/s]
Resolving data files: 100% 41/41 [00:00<00:00, 25300.35it/s]
[done] wikipedia: 500 MB -> /content/brainai/data/corpus_wikipedia.txt
README.md: 100% 1.06k/1.06k [00:00<00:00, 2.87MB/s]
[done] tinystories: 500 MB -> /content/brainai/data/corpus_tinystories.txt
README.md: 100% 1.30k/1.30k [00:00<00:00, 3.38MB/s]
Resolving data files: 100% 54/54 [00:00<00:00, 23182.44it/s]
[done] code: 500 MB -> /content/brainai/data/corpus_code.txt
README.md: 100% 4.80k/4.80k [00:00<00:00, 13.2MB/s]
Resolving data files: 100% 114/114 [00:00<00:00, 14062.01it/s]
[done] math: 500 MB -> /content/brainai/data/corpus_math.txt
[done] wiki_hindi: 500 MB -> /content/brainai/data/corpus_wiki_hindi.txt
Resolving data files: 100% 17/17 [00:00<00:00, 21189.65it/s]
[done] wiki_fren

## 2. Sensor data (images, audio, video)

In [4]:
!python tools/fetch_sensors.py --out $D/sensors --n 20000
!ln -sfn $D/sensors sensors

README.md: 100% 3.90k/3.90k [00:00<00:00, 12.4MB/s]
dataset_infos.json: 100% 3.52k/3.52k [00:00<00:00, 11.4MB/s]

data/train-00000-of-00001-1359597a978bc4(…): downloading bytes:  66% 96.9M/146M [00:01<00:00, 101MB/s, 7.87MB/s  ] 
data/train-00000-of-00001-1359597a978bc4(…): reconstructing file:  30% 44.4M/146M [00:01<00:03, 31.0MB/s]
data/train-00000-of-00001-1359597a978bc4(…): downloading bytes:  84% 123M/146M [00:01<00:00, 110MB/s, 10.2MB/s  ] 
data/train-00000-of-00001-1359597a978bc4(…): downloading bytes:  94% 138M/146M [00:01<00:00, 117MB/s, 11.4MB/s  ]
data/train-00000-of-00001-1359597a978bc4(…): downloading bytes: 100% 138M/138M [00:01<00:00, 77.4MB/s, 12.8MB/s  ]
data/train-00000-of-00001-1359597a978bc4(…): reconstructing file: 100% 146M/146M [00:01<00:00, 82.4MB/s, 13.8MB/s  ]

data/valid-00000-of-00001-70d52db3c749a9(…): downloading bytes:  43% 6.29M/14.6M [00:00<00:00, 12.0MB/s]
data/valid-00000-of-00001-70d52db3c749a9(…): downloading bytes: 100% 13.7M/13.7M [00:00<00:00, 21

In [1]:
import shutil
import os
import glob

# Source directory (large BrainAI)
SRC_DIR = '/content/BrainAI/data'

# Destination directory (smaller brainai)
DEST_DIR = '/content/brainai/data'
os.makedirs(DEST_DIR, exist_ok=True)

# Find all txt files in source
source_files = glob.glob(os.path.join(SRC_DIR, '*.txt'))

if not source_files:
    print(f"No .txt files found in {SRC_DIR}!")
else:
    for file_path in source_files:
        filename = os.path.basename(file_path)
        dest_path = os.path.join(DEST_DIR, filename)

        # Remove existing destination file or symlink to ensure a clean copy
        if os.path.islink(dest_path) or os.path.exists(dest_path):
            try:
                os.remove(dest_path)
            except OSError:
                pass

        # Copy the physical file
        shutil.copy2(file_path, dest_path)
        print(f"Successfully copied: {filename} -> {DEST_DIR}/")

Successfully copied: BrainAI_Batch6_Grade11_Science.txt -> /content/brainai/data/
Successfully copied: language_urdu_large.txt -> /content/brainai/data/
Successfully copied: Aerodynamics_Flight_Dynamics_Control_10000_examples.txt -> /content/brainai/data/
Successfully copied: language_konkani_large.txt -> /content/brainai/data/
Successfully copied: BrainAI_Batch8_Grade11_Commerce.txt -> /content/brainai/data/
Successfully copied: BrainAI_English_KG12_Language_Corpus.txt -> /content/brainai/data/
Successfully copied: BrainAI_Hindi_KG12_Language_Corpus.txt -> /content/brainai/data/
Successfully copied: corpus_math.txt -> /content/brainai/data/
Successfully copied: BrainAI_Konkani_KG12_DEEP_120PLUS_Corpus.txt -> /content/brainai/data/
Successfully copied: marathi_5000_examples.txt -> /content/brainai/data/
Successfully copied: BrainAI_Arabic_KG12_Language_Corpus.txt -> /content/brainai/data/
Successfully copied: urdu_5000_examples.txt -> /content/brainai/data/
Successfully copied: BrainAI

KeyboardInterrupt: 

In [3]:
!cp -r /content/brainai/* /content/BrainAI/
print("Done! Entire brainai folder has been copied and merged into BrainAI.")

cp: '/content/brainai/robot_data' and '/content/BrainAI/robot_data' are the same file
cp: '/content/brainai/sensors' and '/content/BrainAI/sensors' are the same file
Done! Entire brainai folder has been copied and merged into BrainAI.


## 2b. Robot data (state, actions, camera) - LeRobot format

In [5]:
!python tools/fetch_robot.py --out $D/robot_data --skip koch_pick_place_1_lego
!python tools/fetch_robot.py --out $D/robot_data --humanoid          # Unitree H1, ~1.4 GB
# optional, 9-18 GB each: !python tools/fetch_robot.py --out $D/robot_data --humanoid --humanoid-big
!ln -sfn $D/robot_data robot_data

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0% 0/8 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

Reconstructing (incomplete total...):   0% 0.00/7.67M [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/7.67M [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/7.67M [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/7.67M [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/7.68M [00:00<?, ?B/s]

Fetching 8 files:  12% 1/8 [00:00<00:01,  6.55it/s]
Reconstructing (incomplete total...):   0% 4.37k/7.68M [00:00<04:25, 28.9kB/s]
Reconstructing (incomplete total...):   0% 6.63k/7.68M [00:00<04:25, 28.9kB/s]
Reconstructing (incomplete total...):   0% 10.2k/7.69M [00:00<04:25, 28.9kB/s]

Fetching 8 files:  62% 5/8 [00:00<00:00,  7.18it/s]

Fetching 8 files: 100% 8/8 [00:00<0

## 2c. Train the perception encoders (held-out accuracy is printed vs chance)

In [1]:
# 1. Clear any corrupted sensor directory and ensure it is properly copied
!rm -rf /content/BrainAI/sensors
!mkdir -p /content/BrainAI/sensors
!cp -r /content/brainai/sensors/* /content/BrainAI/sensors/

# 2. Re-download if files are invalid, incomplete, or corrupted
import os
import zipfile
for f in ['images.npz', 'audio_sounds.npz', 'video.npz']:
    path = f"/content/BrainAI/sensors/{f}"
    if not os.path.exists(path) or os.path.getsize(path) < 1000:
        print(f"{f} is missing or corrupted. Refetching sensor data...")
        !python /content/BrainAI/tools/fetch_sensors.py --out /content/brainai/sensors --n 10000
        !cp -r /content/brainai/sensors/* /content/BrainAI/sensors/
        break
    try:
        with zipfile.ZipFile(path) as z:
            z.testzip()
    except Exception as e:
        print(f"{f} is invalid ({e}). Refetching...")
        !python /content/BrainAI/tools/fetch_sensors.py --out /content/brainai/sensors --n 10000
        !cp -r /content/brainai/sensors/* /content/BrainAI/sensors/
        break

# 3. Empty CUDA memory cache completely
import torch
torch.cuda.empty_cache()

# 4. Run the sensor training with safe limits to avoid any OOM issues
%cd /content/BrainAI
for m, s in [('images', 500), ('audio', 500), ('video', 500)]:
    !PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python train_sensors.py --modality $m --steps $s --batch 8 --scale small

/content/BrainAI
images: 20000 samples, 200 classes, train 18000 / val 2000, device cuda
step     0  loss 5.606  val_acc 0.005  (8s)
step    50  loss 5.562  val_acc 0.009  (17s)
step   100  loss 5.320  val_acc 0.004  (28s)
step   150  loss 5.824  val_acc 0.007  (40s)
step   200  loss 5.146  val_acc 0.007  (49s)
step   250  loss 5.563  val_acc 0.004  (60s)
step   300  loss 5.422  val_acc 0.004  (71s)
step   350  loss 5.366  val_acc 0.004  (81s)
step   400  loss 5.246  val_acc 0.006  (92s)
step   450  loss 5.147  val_acc 0.009  (103s)
step   499  loss 5.318  val_acc 0.015  (112s)
FINAL images: val top-1 0.015   chance 0.005
audio: 2000 samples, 50 classes, train 1800 / val 200, device cuda
[W1108 09:01:42.725063599 CUDACachingAllocator.cpp:528] expandable_segments: memory mapping failed with OOM on device 0 while trying to map 20971520 bytes (free: 12386304, total: 15637086208).
[W1108 09:01:42.857702344 CUDACachingAllocator.cpp:528] expandable_segments: memory mapping failed with OOM on

## 2d. Behaviour cloning (state -> action); compares to copy-state baseline

In [2]:
# Execute robot behavior cloning pointing to the correct dataset directory
import torch
torch.cuda.empty_cache()
import glob

for d in sorted(glob.glob('/content/BrainAI/robot_data/*')):
    !PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python /content/BrainAI/train_robot.py --dataset $d --steps 1500 --batch 16 --scale small

aloha_sim_insertion_human: 50 episodes (45 train / 5 val), 22500 train frames, state 14-d, action 14-d, device cuda
step     0  train 1.8231  val 1.5974  (1s)
step   150  train 0.0392  val 0.0457  (6s)
step   300  train 0.0241  val 0.0318  (13s)
step   450  train 0.0117  val 0.0308  (19s)
step   600  train 0.0165  val 0.0234  (26s)
step   750  train 0.0136  val 0.0186  (32s)
step   900  train 0.0062  val 0.0183  (37s)
step  1050  train 0.0070  val 0.0153  (44s)
step  1200  train 0.0118  val 0.0178  (50s)
step  1350  train 0.0080  val 0.0159  (57s)
step  1499  train 0.0065  val 0.0163  (63s)
FINAL val MSE 0.0163 | mean-action baseline 1.0960 | copy-state baseline 0.2036
aloha_sim_transfer_cube_human: 50 episodes (45 train / 5 val), 18000 train frames, state 14-d, action 14-d, device cuda
step     0  train 1.6644  val 1.6337  (1s)
step   150  train 0.0332  val 0.0523  (8s)
step   300  train 0.0353  val 0.0390  (13s)
step   450  train 0.0139  val 0.0365  (20s)
step   600  train 0.0213  va

## 3. Distil Qwen into BrainAI (text; teacher writes, student trains)
Byte tokenizer != Qwen BPE, so logit-KL is impossible; we use sequence distillation.

In [ ]:
!python /content/BrainAI/distill.py generate --teacher Qwen/Qwen2.5-1.5B-Instruct --samples 50000

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 338/338 [00:18<00:00, 18.62it/s]
teacher: Qwen/Qwen2.5-1.5B-Instruct
generating 50,000 samples across 20 domains -> data/teacher_corpus.jsonl



## 4. Train (text + distilled). Start with `small`, move to `base` once loss curves look sane.

In [ ]:
!python train_multidomain.py --scale small --steps 40000 --batch 32 --seq 256

In [ ]:
# Local disk is wiped when the session ends: download the trained weights now.
!zip -qr brainai_checkpoints.zip checkpoints
from google.colab import files
files.download('brainai_checkpoints.zip')